# Do LLMs Verify or Conform?
**One-page visual walkthrough** — aws-c-common, 83 functions, gpt-oss-120b & Claude

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10

---
## What information did we give the LLM?

Each **condition** changes exactly one input to the LLM. Everything else is the same feedback loop (CBMC result → LLM → new harness, up to 15 iterations).

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.axis('off')

# columns: Cond | CBMC result shown | Extra info given | Purpose
rows = [
    ('G',      'None (single pass)',    '—',                              '—',                         'Baseline: no feedback at all'),
    ('H',      'CBMC stdout only',      '—',                              '—',                         'Does sacrifice emerge without any prompt guidance?'),
    ('A',      'CBMC stdout only',      '—',                              '—',                         'Standard loop (our main condition)'),
    ('I',      'CBMC stdout only',      'Assertion category on FAIL',     '—',                         'Does knowing the category stop deletion?'),
    ('J',      'CBMC stdout only',      'History of past deletions',      '—',                         'Is sacrifice caused by the LLM "forgetting"?'),
    ('K',      'CBMC stdout only',      'NL contract written first',       '—',                         'Does a pre-written spec reduce sacrifice?'),
    ('Oracle', 'CBMC stdout only',      'GT __CPROVER_assume() provided', '—',                         'Laziness: perfect assumes → weaker asserts?'),
    ('M',      'CBMC stdout only',      'CBMC tip: bound scalar vars',    '—',                         'Fix CBMC knowledge gap → eliminate UNKNOWN'),
]

colors_row = ['#95a5a6','#e67e22','#3498db','#9b59b6','#1abc9c','#e74c3c','#c0392b','#2ecc71']

col_x  = [0.02, 0.07, 0.25, 0.50, 0.55]
headers = ['Cond', 'CBMC feedback', 'Extra info given to LLM', '', 'Purpose / research question']

# Header row
ax.add_patch(mpatches.FancyBboxPatch((0, 0.87), 1.0, 0.11,
    boxstyle='round,pad=0.005', facecolor='#2c3e50', transform=ax.transAxes))
for x, h in zip(col_x, headers):
    ax.text(x, 0.945, h, fontsize=10, fontweight='bold', color='white',
            transform=ax.transAxes, va='center')

row_h = 0.112
for i, (cond, fb, extra, _, purpose) in enumerate(rows):
    y = 0.86 - i * row_h
    color = colors_row[i]
    ax.add_patch(mpatches.FancyBboxPatch((0, y - 0.005), 1.0, row_h * 0.93,
        boxstyle='round,pad=0.003', facecolor=color + '18', transform=ax.transAxes,
        linewidth=0.7, edgecolor=color))
    yc = y + row_h * 0.42
    ax.text(col_x[0], yc, cond, fontsize=11, fontweight='bold', color=color,
            transform=ax.transAxes, va='center')
    ax.text(col_x[1], yc, fb, fontsize=8.5, color='#333',
            transform=ax.transAxes, va='center')
    ax.text(col_x[2], yc, extra, fontsize=8.5, color='#333',
            transform=ax.transAxes, va='center')
    ax.text(col_x[4], yc, purpose, fontsize=8.5, color='#555', style='italic',
            transform=ax.transAxes, va='center')

ax.set_title('Experimental conditions: what changes between them',
             fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## What happens inside the loop? — Sacrifice

When CBMC returns **UNKNOWN** (state space too large), the LLM deletes assertions to make CBMC terminate. These deletions are the core phenomenon.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. What triggers deletions?
ax = axes[0]
ax.pie([92.3, 4.8, 2.9],
       labels=['UNKNOWN-triggered\n(sacrifice)', 'FAIL-triggered\n(legit fix)', 'compile error'],
       colors=['#e74c3c', '#2ecc71', '#f39c12'],
       autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
ax.set_title('Cond A: what triggers deletions?', fontweight='bold')

# 2. Sacrifice rate across conditions
ax = axes[1]
conds = ['G', 'H', 'A', 'I', 'J', 'K', 'Oracle', 'M']
sacr  = [0.0, 86.3, 92.3, 92.7, 93.0, 25.0, 11.8, 0.0]
cols  = ['#95a5a6','#e67e22','#3498db','#9b59b6','#1abc9c','#e74c3c','#c0392b','#2ecc71']
bars = ax.bar(conds, sacr, color=cols, alpha=0.85, width=0.6)
for bar, v in zip(bars, sacr):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v + 1, f'{v:.0f}%',
                ha='center', fontsize=8.5, fontweight='bold')
ax.set_ylabel('Sacrifice rate (%)')
ax.set_title('Sacrifice rate by condition\n(% deletions triggered by UNKNOWN)', fontweight='bold')
ax.set_ylim(0, 108)
ax.grid(True, alpha=0.3, axis='y')

# 3. Deletion scope: panic vs targeted
ax = axes[2]
conds4 = ['A', 'I', 'J', 'K']
panic    = [72, 71, 76, 20]
targeted = [17, 12, 10, 80]
x = np.arange(4); w = 0.35
ax.bar(x - w/2, panic,    width=w, label='Panic (≥3 deleted)', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, targeted, width=w, label='Targeted (1 deleted)', color='#2ecc71', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(conds4)
ax.set_ylabel('%')
ax.set_title('Panic vs targeted deletion\n(K: NL contract acts as anchor)', fontweight='bold')
ax.legend(fontsize=8.5); ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('The sacrifice phenomenon: LLM removes correct assertions to satisfy CBMC',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Core finding: PASS rate ≠ Recall

**PASS rate** = fraction of functions where the LLM harness passes CBMC (verifier satisfied).  
**Recall** = fraction of GT (expert-written) assertions covered by the LLM harness (specification quality).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data = [
    ('G',      31.3, 0.290, '#95a5a6'),
    ('H',      62.7, 0.303, '#e67e22'),
    ('A',      62.5, 0.346, '#3498db'),
    ('K',      81.9, 0.268, '#e74c3c'),
    ('Oracle', 84.3, 0.251, '#c0392b'),
    ('M',      75.3, 0.389, '#2ecc71'),
]

# Left: ranked by PASS
ax = axes[0]
sorted_pass = sorted(data, key=lambda x: x[1], reverse=True)
ax.barh([d[0] for d in sorted_pass], [d[1] for d in sorted_pass],
        color=[d[3] for d in sorted_pass], alpha=0.85, height=0.6)
for d in sorted_pass:
    ax.text(d[1] + 0.5, d[0], f'{d[1]:.1f}%', va='center', fontsize=9)
ax.set_xlabel('PASS Rate (%)')
ax.set_title('① Ranked by PASS rate\n(Oracle best, M third)', fontweight='bold')
ax.set_xlim(0, 100)

# Right: ranked by Recall
ax = axes[1]
sorted_rec = sorted(data, key=lambda x: x[2], reverse=True)
ax.barh([d[0] for d in sorted_rec], [d[2]*100 for d in sorted_rec],
        color=[d[3] for d in sorted_rec], alpha=0.85, height=0.6)
for d in sorted_rec:
    ax.text(d[2]*100 + 0.3, d[0], f'{d[2]:.3f}', va='center', fontsize=9)
ax.set_xlabel('Recall (% of GT assertions covered)')
ax.set_title('② Ranked by Recall\n(M best, Oracle worst)', fontweight='bold')
ax.set_xlim(0, 45)

fig.suptitle(
    'Rankings completely reversed!\n'
    'Oracle: PASS 84.3% (best) vs Recall 25.1% (worst)   |   '
    'M: PASS 75.3% (3rd) vs Recall 38.9% (best)',
    fontsize=11, fontweight='bold', color='#c0392b')
plt.tight_layout()
plt.show()

---
## RQ2: Do missed assertions let real bugs escape? — Mutation Oracle

2,584 synthetic bugs injected into the 83 functions. A bug is **silenced** when GT harness catches it (FAIL) but LLM harness does not (SUCCESS).

**Two metrics**:
- **Sil/GT**: silenced ÷ 351 canonical GT-FAILs (cross-condition comparable)
- **Oracle Recall**: (GT-FAIL ∩ LLM-FAIL) / GT-FAIL per run (within-condition detection rate)

In [ ]:
GT_FAIL = 351

oracle_data = [
    # (label, silenced, oracle_recall_or_None, color)
    ('Cond.\nOracle',  120, 0.453, '#c0392b'),
    ('A\n(gptoss)',     26, None,  '#3498db'),   # old script — not comparable
    ('H\n(gptoss)',     23, 0.488, '#e67e22'),
    ('M\n(gptoss)',     27, 0.680, '#2ecc71'),
    ('G\n(gptoss)',      1, 0.445, '#95a5a6'),
    ('A\n(Claude)',     14, 0.943, '#8e44ad'),
    ('M\n(Claude)',     11, 0.962, '#16a085'),
]

labels   = [d[0] for d in oracle_data]
sil_pct  = [100 * d[1] / GT_FAIL for d in oracle_data]
rec_pct  = [100 * d[2] if d[2] else 0 for d in oracle_data]
colors   = [d[3] for d in oracle_data]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
bars = ax.bar(labels, sil_pct, color=colors, alpha=0.85, width=0.6, edgecolor='white')
for bar, pct, d in zip(bars, sil_pct, oracle_data):
    ax.text(bar.get_x() + bar.get_width()/2, pct + 0.4,
            f'{pct:.1f}%\n({d[1]})', ha='center', fontsize=8, fontweight='bold')
ax.set_ylabel(f'Silenced / {GT_FAIL} canonical GT-FAILs (%)')
ax.set_title('Sil/GT — lower is better\n(how many bugs escape the LLM harness)', fontweight='bold')
ax.set_ylim(0, 44)
ax.axhline(34.2, color='#c0392b', linestyle='--', alpha=0.35,
           linewidth=1.2, label='Cond. Oracle ceiling 34.2%')
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.25, axis='y')

ax = axes[1]
bars2 = ax.bar(labels, rec_pct, color=colors, alpha=0.85, width=0.6, edgecolor='white')
for bar, r, d in zip(bars2, rec_pct, oracle_data):
    if d[2] is None:
        ax.text(bar.get_x() + bar.get_width()/2, 1.5, 'old script\n(n/a)',
                ha='center', va='bottom', fontsize=7.5, color='#e74c3c', style='italic')
    else:
        ax.text(bar.get_x() + bar.get_width()/2, r + 0.8,
                f'{r:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Oracle Recall (%) = LLM-FAIL / GT-FAIL')
ax.set_title('Oracle Recall — higher is better\n(fraction of GT-detectable bugs LLM also catches)', fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.25, axis='y')

patches = [
    mpatches.Patch(color='#3498db', alpha=0.85, label='gptoss-120b'),
    mpatches.Patch(color='#8e44ad', alpha=0.85, label='Claude'),
    mpatches.Patch(color='#c0392b', alpha=0.85, label='Cond. Oracle (GT assumes given)'),
]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9.5,
           bbox_to_anchor=(0.5, -0.04))
fig.suptitle(f'RQ2 Mutation Oracle (canonical GT-FAIL = {GT_FAIL})',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.subplots_adjust(bottom=0.14)
plt.show()

---
## Why were bugs silenced? — Attribution

Three mechanisms explain why an LLM harness fails to catch a bug:

| | **KG** (Knowledge Gap) | **SAC** (Sacrifice) | **AOC** (Assume Over-Constraint) |
|---|---|---|---|
| **What** | LLM never generated the assertion | LLM generated it, then deleted it under UNKNOWN pressure | LLM's `__CPROVER_assume` too tight → mutant path unreachable |
| **Fixable by** | Better model / spec knowledge | Fixing feedback signal (Cond. M) | Looser assumes / vacuity check |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (model, kg, sac, aoc, unk, color, n) in zip(axes, [
    ('gptoss A  (26 silenced)',  84.6, 0.0,  15.4, 0.0,  '#3498db', 26),
    ('Claude A  (14 silenced)',   7.1, 42.9, 28.6, 21.4, '#8e44ad', 14),
]):
    sizes  = [kg, sac, aoc, unk]
    labels = [f'KG\n{kg:.1f}%', f'SAC\n{sac:.1f}%', f'AOC\n{aoc:.1f}%', f'Unknown\n{unk:.1f}%']
    colors = ['#3498db', '#e74c3c', '#f39c12', '#bdc3c7']
    non_zero = [(s, l, c) for s, l, c in zip(sizes, labels, colors) if s > 0]
    ax.pie([x[0] for x in non_zero],
           labels=[x[1] for x in non_zero],
           colors=[x[2] for x in non_zero],
           autopct='%1.0f%%', startangle=90,
           textprops={'fontsize': 9.5})
    ax.set_title(model, fontsize=12, fontweight='bold', color=color)

fig.suptitle(
    'Silenced mutant attribution by mechanism\n'
    'gptoss: mostly knowledge gap (KG) — never knew the property existed\n'
    'Claude: mostly sacrifice (SAC) + AOC — knew it but removed it under CBMC pressure',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Finding | Evidence |
|---------|----------|
| **LLMs sacrifice correct assertions under CBMC UNKNOWN** | 92.3% of deletions are UNKNOWN-triggered, not FAIL-triggered |
| **Sacrifice is emergent, not instructed** | Cond. H (no guidance) still has 86.3% sacrifice rate |
| **Sacrifice is deliberate** | Cond. I (told the category): 92.7%; LLM knows and still deletes |
| **PASS rate ≠ specification quality** | Oracle (84.3% PASS) = worst recall; M (75.3% PASS) = best recall |
| **Missed assertions let bugs escape** | 26 silenced (gptoss), 14 silenced (Claude) out of 351 GT-FAIL mutants |
| **Stronger model reduces KG, but SAC+AOC persist** | Claude KG drops from 84.6% → 7.1%; but still 14 silenced via SAC/AOC |
| **Fix: repair the feedback signal, not the model** | Cond. M: one CBMC tip → sacrifice = 0, recall = highest |

**Central claim**: LLMs optimise for verifier satisfaction, not specification completeness.  
The feedback mechanism itself creates the pressure — fixing the signal (Condition M) is more effective than upgrading the model.